# EasyAgent 权限系统与中断机制 (Permission Engine & Interruption)

这个示例展示了如何在 EasyAgent 中定义权限系统规则，以及如何在使用大模型调用高风险工具时触发和捕获 `ToolConfirmationRequired` 中断 (Interruption)。


In [ ]:
# 环境与基础初始化
import os, sys
from pathlib import Path

project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from core.llm import EasyLLM
from agent.BasicAgent import BasicAgent
from Tool.ToolRegistry import ToolRegistry
from Tool.builtin import register_shell_tools, register_file_write_tool
from core.permissions import PermissionMode, PermissionRule, PermissionBehavior
from core.Exception import ToolConfirmationRequired

llm = EasyLLM(
    provider="openai",
    base_url="http://127.0.0.1:5124/v1",
    api_key="122",
    model="qwen3.5-9b",
)
registry = ToolRegistry()
register_shell_tools(registry, workspace_root=project_root)
register_file_write_tool(registry, workspace_root=project_root)

print("✅ 基础环境, 工具 (Bash, FileWrite) 注册完成。")


ValueError: API密钥必须被提供或在.env文件中定义。

In [ ]:
# 1. 定义与初始化带权限配置的 Agent
agent = BasicAgent(
    name="SecurityAwareAgent",
    llm=llm,
    tool_registry=registry,
    enable_tool=True,
)

# 我们通过设定一个权限规则：任何对 Bash 工具的调用都必须经过用户(User)确认 (ASK)
# 优先级数字越大优先级越高
agent.set_permission_rules(
    source="demo_guard",
    rules=[
        PermissionRule(
            tool_name="Bash",
            behavior=PermissionBehavior.ASK,
            description="所有的终端系统命令必须经过人工确认防范风险。"
        )
    ],
    priority=100
)

print(f"✅ Agent 的当前权限模式为: {agent.permission_context.mode.value}")
print(f"✅ 已挂载针对 Bash 的 ASK 规则。")


In [ ]:
# 2. 展示中断机制 (Interruption Catching)

query = "请帮我用 Bash 命令行查看一下当前目录下有哪些文件。"

print("=== 开始调用模型执行任务 ===")
try:
    # 当 agent 企图调用 Bash 时，由于被命中 ASK 规则
    # 调用链底层的 PermissionEngine 会返回 Ask 裁决，从而向外抛出 ToolConfirmationRequired
    res = agent.invoke(query, max_iter=3)
    print("模型正常结束:", res)

except ToolConfirmationRequired as e:
    print("\n==========================")
    print("🚨 [捕获到权限中断] 🚨")
    print(f"拦截原因: {e.args[0]}")
    print(f"模型企图调用的工具: {e.tool_name}")
    print(f"模型使用的参数: {e.tool_args}")
    print("==========================\n")
    print("在真实的控制台或者 GUI UI 中，这会产生一个类似 [YES / NO] 的弹窗，将执行流挂起。")


In [ ]:
# 3. 如果放宽权限 (Accept Edits / Bypass)
# 可以利用 set_permission_mode 直接允许某些类型的风险操作
agent.set_permission_mode(PermissionMode.BYPASS)
print(f"\n✅ 已经切换模式为: {agent.permission_context.mode.value}")

print("=== 在 BYPASS 下重新运行命令 ===")
try:
    res = agent.invoke(query, max_iter=3)
    print("\n✅ 模型成功执行并在 BYPASS 模式下绕过了安全询问：\n", res)
except Exception as e:
    print("发生异常:", e)
